In [ ]:
#----IMPORT LIBRAIRIES----
import pandas as pd
import seaborn as sns
import json
import math
from scipy.stats import zscore

import plotly
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import plotly.io as pio
from plotly.subplots import make_subplots

import pvlib

import mlflow
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

from mlflow.models.signature import infer_signature

import boto3

from dotenv import load_dotenv
import os

load_dotenv()
# os.environ["MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING"] = "false"

import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split, Dataset
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go

import Model_func as mf
import func_cleaning as fc

In [ ]:
#---VARIABLES----
weather_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/openweathermap/merge_openweathermap_cleaned.csv'
solar_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/solar/raw_solar_data.csv'
landsat_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/LandSat/result_EarthExplorer_region_ARA.csv'

col_solar = ['Time', 'Ap', '10cm', 'K index Planetary'] # ALWAYS include a 'Time' column (used to merge datasets)
prod_data_path = 'https://renergies99-bucket.s3.eu-west-3.amazonaws.com/public/prod/eCO2mix_RTE_Auvergne-Rhone-Alpes_cleaned.csv'
target = 'tch_solaire_(%)'

col_solar = ['Time', 'Ap', '10cm', 'K index Planetary'] # ALWAYS include a 'Time' column (used to merge datasets)
cities_list = ['Moulins', 'Annecy', 'Nyons', 'Saint-Étienne', 'Aurillac']

In [ ]:
#--- Data Collection ----
full_dataset = fc.create_full_dataset(weather_data_path, solar_data_path, landsat_data_path, prod_data_path, 
                                   cities_list, col_solar, target)

# full_dataset.to_csv('../../../Mes_fichiers_vrac/data_csv.csv')


In [ ]:
#--- Data cleaning ---
df = full_dataset.copy()
print(f'df shape: {df.shape}')

# gestion des Nan
df_no_Nan = fc.handle_nan(df)
print(f'df_no_Nan shape: {df_no_Nan.shape}')

# clean data (convert int to float, select type columns, remove unique values)
df_clean = fc.clean_dataframe(df_no_Nan, type='numeric')
print(f'df_clean shape: {df_clean.shape}')

#suppression des outliers
df_no_outliers = fc.remove_outliers(df_clean, target, method='iqr')
print(f'df_no_outliers shape: {df_no_outliers.shape}')


In [ ]:
#--- Features and target definition
X = df_no_outliers.drop(target, axis=1)
y = df_no_outliers[target]

tensor_X = torch.tensor(X.astype(float).values)
tensor_y = torch.tensor(y.astype(float).values)



In [ ]:
X.astype(float)

In [ ]:
EXPERIMENT_NAME = "NN_model"
run_name = 'eruptive_brain'
# threshold=[42]

#---Preprocess
# result_preprocess = mf.preprocessing_and_pipeline(X, LinearRegression(), suffixes=['humidity'], split_thresholds=threshold)
# pipeline = result_preprocess["pipeline"]
# preprocessor = result_preprocess["preprocessor"]

class CustomDataset(Dataset):
    def __init__(self, target, X, transform=None, target_transform=None):
        self.targets = target
        self.datas = X
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.datas)

    def __getitem__(self, idx):
        # img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        data = self.datas[idx, :]
        label = self.targets[idx]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return data.to(torch.float32), label.to(torch.float32)

dataset = CustomDataset(tensor_y, tensor_X)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=300, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=300, shuffle=True)

# x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=24)
# input_example = x_train.iloc[:3]

In [ ]:
sample_data, sample_target = next(iter(train_loader))

In [ ]:
X.shape

In [ ]:
class NNModel(nn.Module):
    def __init__(self):
        super(NNModel, self).__init__()
        self.norm = nn.BatchNorm1d(82)
        self.fc1 = nn.Linear(82, 82)
        self.fc2 = nn.Linear(82, 16)
        self.fc3 = nn.Linear(16, 8)
        self.relu = nn.ReLU()
        self.fc_final = nn.Linear(8,1)

    def forward(self, x):
        # x = x.view(x.size(0), -1)  # Flatten the input
        x = x.to(torch.float32)
        x = self.norm(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc_final(self.relu(self.fc3(x)))
        return x

model = NNModel()
print(model)

In [ ]:
criterion = nn.MSELoss()
base_lr = 0.03
optimizer = optim.Adam(model.parameters(), lr=base_lr)

In [ ]:
from torchinfo import summary
summary(model,input_data=sample_data)

In [ ]:
#---MLFlow params
os.environ["APP_URI"] = "https://renergies99-mlflow.hf.space/"

mlflow.set_tracking_uri(os.environ["APP_URI"])
mlflow.set_experiment(EXPERIMENT_NAME)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

mlflow.pytorch.autolog()  # enables automatic logging for scikit-learn

In [ ]:
def train(model, train_loader, val_loader, criterion, optimizer, epochs=100):
    """
    Function to train a PyTorch model with training and validation datasets.
    
    Parameters:
    model: The neural network model to train.
    train_loader: DataLoader for the training dataset.
    val_loader: DataLoader for the validation dataset.
    criterion: Loss function (e.g., Binary Cross Entropy for classification).
    optimizer: Optimization algorithm (e.g., Adam, SGD).
    epochs: Number of training epochs (default=100).
    
    Returns:
    history: Dictionary containing loss and accuracy for both training and validation.
    """
    
    # Dictionary to store training & validation loss and accuracy over epochs
    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
    outputs_dic = {'pred': [], 'target': []}
    
    for epoch in range(epochs):  # Loop over the number of epochs
        model.train()  # Set model to training mode
        total_loss, correct = 0, 0  # Initialize total loss and correct predictions
        
        # Training loop
        for inputs, labels in train_loader:
            optimizer.zero_grad()  # Reset gradients before each batch
            outputs = model(inputs).squeeze()  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            loss.backward()  # Backpropagation (compute gradients)
            optimizer.step()  # Update model parameters
            
            total_loss += loss.item()  # Accumulate batch loss
            # train_mae = mean_absolute_error(labels, outputs)
            # mse = mean_squared_error(y_test, y_pred)
            outputs_dic['pred'].append(outputs.detach().numpy().reshape(-1).tolist())
            outputs_dic['target'].append(labels.detach().numpy().reshape(-1).tolist())

        # correct += (torch.argmax(outputs,dim=1) == labels).sum().item()  # Count correct predictions
        if loss.item() > 5:
            new_lr = base_lr / (1 + loss.item())
        else: 
            new_lr = base_lr*loss.item() / 100

        for g in optimizer.param_groups:
            g['lr'] = new_lr

        # Compute average loss and accuracy for training
        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_loader.dataset)
        
        # Validation phase (without gradient computation)
        model.eval()  # Set model to evaluation mode
        val_loss, val_correct = 0, 0
        

        with torch.no_grad():  # No need to compute gradients during validation
            for inputs, labels in val_loader:
                outputs = model(inputs).squeeze()  # Forward pass
                loss = criterion(outputs, labels)  # Compute loss
                val_loss += loss.item()  # Accumulate validation loss
                # val_correct += (torch.argmax(outputs,dim=1) == labels).sum().item()  # Count correct predictions

        # Compute average loss and accuracy for validation
        val_loss /= len(val_loader)
        val_acc = val_correct / len(val_loader.dataset)
        
        # Store metrics in history dictionary
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)
        

        # history['mae'].append(train_mae)
        # history['val_mae'].append(val_mae)
        
        # Print training progress
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    return history, outputs_dic, loss  # Return training history

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name):
# Train the model
    history, outputs_dic, mse = train(model, train_loader, val_loader, criterion, optimizer, epochs=3000)
    # logging metrics
    # mlflow.log_metric("MAE", mae)
    mlflow.log_metric("MSE", mse)
    # mlflow.log_metric("RMSE", rmse)
    # mlflow.log_metric("R2", r2)
    # mlflow.log_metric("Adjusted_R2", adj_r2)

    # Log the full pipeline as a model
    mlflow.pytorch.log_model(model)

In [ ]:
outputs_dic['pred'] = outputs_dic['pred']
outputs_dic['target'] = outputs_dic['target']
test = pd.DataFrame.from_dict(outputs_dic)
px.scatter(test, x='pred', y='target')

In [ ]:
outputs_dic['pred'] = outputs_dic['pred'][-1]
outputs_dic['target'] = outputs_dic['target'][-1]
test = pd.DataFrame.from_dict(outputs_dic)
px.scatter(test, x='pred', y='target')